<a href="https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.
30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#**Task type: Ranking / Scoring**

My lane (CTR / Engagement Opportunity Scoring) is not a yes/no problem, so it is
not classification. Whether a page is an "opportunity" is not binary — the gap
between a page's actual CTR and the CTR expected for its position tier is a
continuous, graded signal: some pages sit barely below expectation, others sit
far below it. Picking one fixed cutoff (e.g. `ctr < 0.5`) would collapse that
gradient into an arbitrary yes/no split, and it wouldn't match the decision I'm
actually supporting: "which page should the editor review first?" — that needs
an ordered list, not a flag. So the natural mapping is ranking/scoring: score
every page by its opportunity gap, then rank so limited reviewer time goes to
the top of the list first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

#**Proxy: impressions-weighted CTR opportunity gap — engineered, not observed as a ready-made column**

The raw signals (`ctr`, `position_tier`, `impressions_90d`) are observed: real, measured
columns in the dataset. But the score I actually rank by does not exist as-is — I build it
myself, in three steps.

**Step 0 — keep only pages with a real position signal.** Pages with `position_tier ==
"no_data"` have `avg_position == 0`, meaning GSC never recorded a ranking for them. Averaging
`ctr` over that group wouldn't mean "normal performance for that rank" — it would just be an
average over pages with no ranking signal at all. These rows are excluded before anything
else is computed, so every tier-level average below only compares pages against others that
actually share a real position.

**Step 1 — the raw gap:**

    expected_ctr(tier) = mean(ctr) across valid pages in that position_tier
    opportunity_gap    = expected_ctr(tier) − ctr

A positive gap means the page underperforms what's normal for its own tier (bigger gap =
bigger opportunity). This stays a continuous score, not a yes/no cutoff — consistent with
the Ranking/Scoring framing from section 1.

**Step 2 — why the raw gap alone isn't enough:** checking `opportunity_gap`'s unique values
showed heavy repetition — thousands of pages shared the exact same gap value. All of them had
`ctr = 0`, which collapses `opportunity_gap` to a constant (`expected_ctr(tier)`) regardless of
how many times the page was actually seen. A page with 5 impressions and a page with 5,000
impressions — both at `ctr = 0` — would get identical scores, even though the second is a far
more reliable, higher-volume opportunity.

**Fix — weight the gap by traffic volume:**

    weighted_score = opportunity_gap × log1p(impressions_90d)

I used `log1p` (not raw `impressions_90d`) because impressions span a huge range (tens to
hundreds of thousands) while the gap only ranges roughly 0–3 — multiplying directly would let
volume dominate the ranking and drown out the gap itself. The log compresses that wide range
(each ×10 increase in impressions becomes a fixed +constant on the log scale) while still
preserving order. `log1p` specifically avoids errors for pages with `impressions_90d = 0`.

**What this score is — and isn't.** `weighted_score` is a practical heuristic: it down-weights
low-volume noise so pages aren't ranked purely on a gap computed from a handful of impressions.
It is **not** a formal statistical confidence measure — it doesn't estimate uncertainty bounds
the way a confidence interval on the true CTR would, it just scales the gap by a rough proxy
for sample size. Sorting by it does surface pages with both a real gap and meaningful traffic,
rather than pages that only look good because of raw volume or a gap that's real but
statistically unreliable due to tiny sample size — but that check is directional, not a proof.

Because I engineered this myself rather than reading a label FlyRank already computed, I have
to be careful it stays a *proxy* for opportunity, not treated as proven ground truth.

In [2]:
import numpy as np

# Step 0: keep only pages with a real position signal
valid = df[df["position_tier"] != "no_data"].copy()

# Step 1: raw gap vs. tier-level expected CTR
expected_ctr = valid.groupby("position_tier")["ctr"].transform("mean")
valid["opportunity_gap"] = expected_ctr - valid["ctr"]

# Step 2: weight by traffic volume so low-impression pages don't dominate/tie
valid["weighted_score"] = valid["opportunity_gap"] * np.log1p(valid["impressions_90d"])

valid[["content_id", "position_tier", "ctr", "impressions_90d",
       "opportunity_gap", "weighted_score"]] \
    .sort_values("weighted_score", ascending=False) \
    .head(10)

,content_id,position_tier,ctr,impressions_90d,opportunity_gap,weighted_score
7678,content_8451fc6f034d,top_3,0.03,272144,1.453611,18.190613
26844,content_8c19996aa890,top_3,0.15,509252,1.333611,17.524576
3331,content_4a6607efcb46,top_3,0.01,128068,1.473611,17.330138
16736,content_e12868d1f396,top_3,0.07,149712,1.413611,16.845255
9412,content_8053a66bd6ac,top_3,0.08,52687,1.403611,15.260254
7860,content_d225ec9f3d46,top_3,0.05,26470,1.433611,14.599610
27107,content_0022a6b4290f,top_3,0.07,29747,1.413611,14.560919
10893,content_6f81ccd92b64,top_3,0.19,73675,1.293611,14.498052
20537,content_f4e210ee0c27,top_3,0.06,24784,1.423611,14.404082
7122,content_7a6df559322d,top_3,0.14,43650,1.343611,14.355110


## 3. Success metric

*One metric you can defend. What number means 'good'?*

#**Metric: Precision@K, K ≈ 25 (assumption)**

Ranking/scoring problems are judged by precision@K: of the top K pages by `weighted_score`,
how many turn out to be genuine opportunities?

**Defining a "hit":** because `weighted_score` is a proxy I engineered (section 2), there is
no ready-made column in the data that says "this page is a genuine opportunity." A hit can
only be defined by an editor manually reviewing a flagged page and confirming it's worth
acting on. That means precision@K is **defined now, but not computable from this static
dataset alone** — it needs either a completed round of human review, or a forward time window
showing CTR actually improved after action (available in the full warehouse, weeks 3+, not in
this 30k-row snapshot).

**Choosing K:** I don't have a real reviewer-capacity number, so I'm assuming one — roughly 5
pages reviewed per day, 5-day work week → K ≈ 25. This is an assumption, not a measured fact,
and I'm stating it as one here rather than treating it as settled.

**What I can do today:** produce the actual top-K list a reviewer would receive, and write the
precision@K function so it's ready to run the moment review labels exist — that's the honest
version of "name the metric before training" when the ground truth doesn't exist yet.

In [5]:
K = 25  # assumption: ~5 pages/day x 5-day work week (see markdown)
top_k[["content_id", "position_tier", "ctr", "impressions_90d", "weighted_score"]]
top_k = valid.sort_values("weighted_score", ascending=False).head(K)



def precision_at_k(is_real_opportunity: pd.Series) -> float:
    """
    is_real_opportunity: one boolean per row of `top_k`, filled in by an editor
    after manually reviewing each flagged page (True = genuine opportunity,
    False = false positive). Not computable from this dataset alone --
    this function is ready to run once that review happens.
    """
    return is_real_opportunity.sum() / len(is_real_opportunity)
print(top_k)

# Example of future use, once review labels exist:
# reviewed = pd.Series([True, True, False, ...])  # one entry per row in top_k
# precision_at_k(reviewed)

                 content_id          client_id  search_volume  competition  \
7678   content_8451fc6f034d  client_d029fa3a95           10.0         1.00   
26844  content_8c19996aa890  client_4e07408562           70.0         0.01   
3331   content_4a6607efcb46  client_6208ef0f77            0.0         0.00   
16736  content_e12868d1f396  client_4e07408562        12100.0         0.01   
9412   content_8053a66bd6ac  client_19581e27de            0.0         0.00   
7860   content_d225ec9f3d46  client_f369cb89fc            0.0         0.00   
27107  content_0022a6b4290f  client_f369cb89fc            0.0         0.00   
10893  content_6f81ccd92b64  client_19581e27de           70.0         0.42   
20537  content_f4e210ee0c27  client_7f2253d7e2            0.0         0.00   
7122   content_7a6df559322d  client_19581e27de           30.0         0.49   
21458  content_9009d5d37434  client_6208ef0f77            0.0         0.00   
22592  content_f9d82e71e363  client_19581e27de           20.0   

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.